# Guitar MIDI: before and after the cleanup

What the tab pipeline (`api/tab/`, the same code behind `POST /tab`) does to the guitar MIDI of a
song, step by step, drawn as a piano roll and **played with a synthesized electric guitar**.

* **Before:** `output/midi/<song>/guitar.mid` as MuScriptor / NeuralNote wrote it.
* **After:** the `auto` part (what one guitarist plays: rhythm, and the lead line in solos) and
  the `rhythm` part (chords, under the solos too).

Kernel: the project venv (`.venv`, Python 3.12). In VS Code / Cursor pick it as the kernel. In
the browser: `uv pip install --python .venv jupyterlab` and `.venv/bin/jupyter lab`.
No Docker, no soundfont: the guitar is a small numpy synth (Karplus-Strong string + amp + cabinet).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "api").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import mido
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from scipy.signal import butter, fftconvolve, lfilter, sosfilt

from api.tab import build_tab
from api.tab.core import TUNINGS, Note
from api.tab.guitar_cleaner import clean_guitar

OUT = ROOT / "output"
songs = sorted(p.parent.name for p in (OUT / "midi").glob("*/guitar.mid"))
for i, s in enumerate(songs):
    print(i, s)

## Settings
`SONG` is a folder of `output/midi/` (a number from the list above also works). `T0`–`T1` is the
window that gets drawn and played (`T1 = None` = to the end of the song).

In [ ]:
SONG = 0            # folder name or index in the list above
T0, T1 = 134, 164   # seconds
DRIVE = 6.0         # amp gain: 0 = clean, 3 = crunch, 6–10 = rock lead
SR = 44100

SONG = songs[SONG] if isinstance(SONG, int) else SONG
MIDI_PATH = OUT / "midi" / SONG / "guitar.mid"
STEM = next(iter(sorted((OUT / "stems" / SONG).glob("guitar.*"))), None)
DRUMS = next(iter(sorted((OUT / "stems" / SONG).glob("drums.*"))), None)    # beat grid for the riffs
VOCALS = next(iter(sorted((OUT / "stems" / SONG).glob("vocals.*"))), None)  # sung notes labelled as guitar
print(MIDI_PATH.relative_to(ROOT), "| stem:", STEM.relative_to(ROOT) if STEM else "none (no loudness estimate)")

## 1 · Load the MIDI (before)

In [ ]:
from api.tab.evaluate import read_midi_notes as read_midi   # keeps each note's guitar (program)

raw = read_midi(MIDI_PATH)
if T1 is None:
    T1 = max(n.end for n in raw)
print(f"{len(raw)} notes, {raw[-1].end:.0f} s, pitches {min(n.midi for n in raw)}–{max(n.midi for n in raw)}")

## 2 · Run the pipeline
`build_tab` is the exact code behind `POST /tab`. The guitar cleaner (`api/tab/guitar_cleaner.py`)
splits the song's guitars by MIDI program, decides chord or notes for every strike, turns chords into
their chart shapes and leaves single notes as they are.

In [ ]:
tab = build_tab(raw, STEM, DRUMS, VOCALS)                          # what the interface shows
tuning = TUNINGS[tab["tuning"]["id"]]
cleaned = clean_guitar([Note(n.time, n.duration, n.midi, n.velocity, n.layer) for n in raw], tuning)
chords_only = [n for s in cleaned.rhythm if s.kind == "chord" for n in s.notes]   # for the plot
after = {name: part["notes"] for name, part in tab["parts"].items()}
solos = [(s["start"], s["end"]) for s in tab["solos"]]

def in_window(ns):
    return [n for n in ns if T0 <= (n["time"] if isinstance(n, dict) else n.time) < T1]

st = tab["stats"]
print(f"tuning: {tab['tuning']['name']}   loudness from: {st['velocity']}   "
      f"sung notes removed: {st['vocal_notes_removed']}")
print(f"guitars (MIDI programs): {st['layers']}   chord strikes: {st['chords']}   note strikes: {st['note_events']}")
print(f"chords: {', '.join(f'{k}×{v}' for k, v in st['chord_names'].items())}")
print(f"solos: {', '.join(f'{a:.0f}–{b:.0f} s' for a, b in solos) or 'none'}\n")
print(f"{'step':<34}{'whole song':>12}{f'{T0:g}–{T1:g} s':>12}")
for name, ns in [("before (MIDI)", raw), ("chords (standard shapes)", chords_only),
                 ("rhythm", after["rhythm"]), ("auto (what gets played)", after["auto"])]:
    print(f"{name:<34}{len(ns):>12}{len(in_window(ns)):>12}")


## 3 · Piano roll
Grey bars are notes that are no longer there in the `auto` part (approximate match: same pitch,
onset within 60 ms; cleanup can move an onset a little or fold a note down an octave).
Shaded areas are the solos the pipeline found. Colour = loudness.

In [ ]:
def as_rows(ns):
    return [(n["time"], n["duration"], n["midi"], n["velocity"]) if isinstance(n, dict)
            else (n.time, n.duration, n.midi, n.velocity) for n in ns]

kept = {}
for t, _, m, _ in as_rows(after["auto"]):
    kept.setdefault(m, []).append(t)
def survives(t, m):
    return any(abs(t - u) < 0.06 for u in kept.get(m, ()))

panels = [("Before: MIDI", raw), ("Chords only (standard shapes)", chords_only),
          ("Rhythm: chords + notes that don't clash", after["rhythm"]),
          ("Auto (what gets played)", after["auto"])]
fig, axes = plt.subplots(len(panels), 1, figsize=(15, 2.8 * len(panels)), sharex=True, sharey=True)
cmap = plt.get_cmap("viridis")
for ax, (title, ns) in zip(axes, panels):
    rows = [r for r in as_rows(ns) if r[0] < T1 and r[0] + r[1] > T0]
    for t, d, m, v in rows:
        lost = title.startswith("Before") and not survives(t, m)
        ax.add_patch(plt.Rectangle((t, m - 0.4), d, 0.8, lw=0,
                                   color="0.75" if lost else cmap(0.15 + 0.85 * min(v, 1))))
    for a, b in solos:
        ax.axvspan(max(a, T0), min(b, T1), color="tab:red", alpha=0.07, lw=0)
    ax.set_title(f"{title} — {len(rows)} notes", loc="left", fontsize=10)
    ax.set_ylabel("MIDI")
    ax.grid(axis="y", alpha=0.25)
lo, hi = min(n.midi for n in raw), max(n.midi for n in raw)
axes[-1].set_xlim(T0, T1); axes[-1].set_ylim(lo - 1, hi + 1); axes[-1].set_xlabel("seconds")
plt.tight_layout(); plt.show()

## 4 · Electric guitar synth
Each note is a plucked string (Karplus-Strong with a fractional-delay all-pass, so it is in tune
up to the high frets), damped when the note ends. The sum then goes through an amp (soft clipping,
`DRIVE`), a 4×12 cabinet (band-pass) and a short room reverb.

In [ ]:
_pluck_cache: dict = {}

def pluck(midi: int, dur: float, sr: int = SR) -> np.ndarray:
    dur = round(min(max(dur, 0.06), 4.0), 2)
    key = (midi, dur)
    if key in _pluck_cache:
        return _pluck_cache[key]
    f = 440.0 * 2 ** ((midi - 69) / 12)
    period = sr / f
    N = int(np.floor(period - 0.5 - 1e-3))
    d = period - N - 0.5                              # fractional delay left for the all-pass
    c = (1 - d) / (1 + d)
    t60 = 4.0 * (110 / f) ** 0.5                       # low strings ring longer
    a = 10 ** (-3 / (t60 * f))
    rng = np.random.default_rng(midi)
    exc = rng.uniform(-1, 1, N)
    exc -= np.roll(exc, max(1, int(N * 0.13)))         # picked at 13 % of the string length
    n = int((dur + 0.12) * sr)
    x = np.zeros(n); x[:N] = exc
    den = np.zeros(N + 3); den[0] = 1; den[1] = c
    den[N:N + 3] -= a * np.convolve([0.5, 0.5], [c, 1])
    y = lfilter([1, c], den, x)
    release = int(dur * sr)                            # fret hand lifts: fast damping
    y[release:] *= np.exp(-np.arange(n - release) / (0.025 * sr))
    _pluck_cache[key] = y
    return y

def amp(x: np.ndarray, drive: float = DRIVE, sr: int = SR) -> np.ndarray:
    x = sosfilt(butter(2, 90, "hp", fs=sr, output="sos"), x)
    x = x / (np.max(np.abs(x)) + 1e-9)
    if drive > 0:
        x = np.tanh(drive * (x + 0.05)) - np.tanh(drive * 0.05)   # slightly asymmetric clipping
    x = sosfilt(butter(4, [80, 4800], "bp", fs=sr, output="sos"), x)   # cabinet
    room = np.random.default_rng(0).standard_normal(int(0.35 * sr)) * np.exp(-np.arange(int(0.35 * sr)) / (0.07 * sr))
    x = x + 0.12 * fftconvolve(x, room)[: len(x)] / np.sqrt(np.sum(room ** 2))
    return 0.9 * x / (np.max(np.abs(x)) + 1e-9)

def render(ns, t0: float = T0, t1: float = T1, drive: float = DRIVE) -> np.ndarray:
    out = np.zeros(int((t1 - t0 + 1.0) * SR))
    for t, d, m, v in as_rows(ns):
        if not (t0 <= t < t1):
            continue
        y = pluck(m, d) * (0.25 + 0.75 * min(v, 1))
        i = int((t - t0) * SR)
        y = y[: len(out) - i]
        out[i:i + len(y)] += y
    return amp(out, drive)

def play(label, audio):
    print(label)
    display(Audio(audio, rate=SR, normalize=False))

## 5 · Listen

In [ ]:
if STEM:
    ref_sr = sf.info(STEM).samplerate
    ref, _ = sf.read(STEM, start=int(T0 * ref_sr), stop=int(T1 * ref_sr), always_2d=True)
    print("Original guitar stem (for reference)")
    display(Audio(ref.T, rate=ref_sr))
play("Before: MIDI as transcribed", render(raw))
play("After: auto part (what the tab shows)", render(after["auto"]))
play("After: rhythm part (chords in standard shapes)", render(after["rhythm"]))

Same thing without distortion, which makes wrong notes much easier to hear:

In [ ]:
play("Before (clean amp)", render(raw, drive=0))
play("After: auto (clean amp)", render(after["auto"], drive=0))

## 6 · Save the cleaned MIDI

In [ ]:
def write_midi(ns, path: Path) -> None:
    mid = mido.MidiFile(ticks_per_beat=480)
    track = mido.MidiTrack(); mid.tracks.append(track)
    track.append(mido.MetaMessage("set_tempo", tempo=500000))       # 120 BPM: 960 ticks per second
    track.append(mido.Message("program_change", program=29))         # overdriven guitar
    events = []
    for t, d, m, v in as_rows(ns):
        events.append((t, 1, m, max(1, min(127, round(v * 127)))))
        events.append((t + d, 0, m, 0))
    last = 0
    for t, on, m, v in sorted(events):
        tick = round(t * 960)
        track.append(mido.Message("note_on" if on else "note_off", note=m, velocity=v, time=tick - last))
        last = tick
    mid.save(path)

for name in ("auto", "rhythm"):
    path = OUT / "midi" / SONG / f"guitar_tab_{name}.mid"
    write_midi(after[name], path)
    print("saved", path.relative_to(ROOT))

## 7 · MuScriptor: all instruments vs. guitar only

By default MuScriptor transcribes every instrument at once (that is how the pipeline and NeuralNote
run it). It can also be **told to transcribe only some instruments**: it then gets the guitar as a
condition and cannot emit any other instrument. Here the window `T0`–`T1` is transcribed both ways
from the same audio, then compared: notes, timing, piano roll, tab and sound.

* `SOURCE = "mix"`: the downloaded song (`output/downloads/`), or the sum of the stems when the
  download isn't there. This is what the pipeline transcribes.
* `SOURCE = "stem"`: the separated guitar stem.

MuScriptor large runs at ~2.7× real time on an M2: a 30 s window takes ~1.5 min per mode. Results
are cached in `output/midi/<song>/compare/`, so re-running the cell is instant.

The tab here is built from the window alone. Solo detection learns the song's usual chords from
what it is given, so on a short window that is mostly solo it is unreliable: compare the
**transcribed** notes first, the tab second.

In [ ]:
import json, os, subprocess, soxr

SOURCE = "mix"     # "mix" or "stem"
MODES = {
    "all instruments": [],
    "guitar only": ["clean_electric_guitar", "distorted_electric_guitar"],
}
ENGINE = ROOT / "engine" / "bin" / "t2s_transcribe"
MODEL = Path(os.getenv("MUSCRIPTOR_MODEL", Path.home() / "Library/NeuralNote/models/muscriptor-large-f16.gguf"))
GUITAR_PROGRAMS = range(24, 32)

def window_audio(source: str) -> tuple[np.ndarray, int]:
    # Mono float audio of the window, at the file's rate
    if source == "stem":
        files = [STEM]
    else:
        download = OUT / "downloads" / f"{SONG}.wav"
        files = [download] if download.exists() else sorted((OUT / "stems" / SONG).glob("*.*"))
    if not files or files[0] is None:
        raise FileNotFoundError(f"no audio for {SONG!r} ({source})")
    if source == "mix" and len(files) > 1:
        print("no download for this song: transcribing the sum of its stems")
    mix, rate = None, None
    for f in files:
        rate = sf.info(f).samplerate
        y, _ = sf.read(f, start=int(T0 * rate), stop=int(T1 * rate), always_2d=True)
        y = y.mean(1)
        mix = y if mix is None else mix[: len(y)] + y[: len(mix)]
    return mix, rate

def transcribe_window(source: str, instruments: list[str]) -> list[Note]:
    cache = OUT / "midi" / SONG / "compare" / f"{source}_{T0:g}-{T1:g}_{'+'.join(instruments) or 'all'}.json"
    if not cache.exists():
        cache.parent.mkdir(parents=True, exist_ok=True)
        y, rate = window_audio(source)
        raw16 = cache.with_suffix(".f32")
        soxr.resample(y, rate, 16000, quality="VHQ").astype(np.float32).tofile(raw16)
        cmd = [str(ENGINE), "--model", str(MODEL), "--audio", str(raw16), "--out", str(cache)]
        if instruments:
            cmd += ["--instruments", ",".join(instruments)]
        proc = subprocess.Popen(cmd, stderr=subprocess.PIPE, text=True)
        for line in proc.stderr:
            if line.startswith("progress"):
                print(f"\r  {float(line.split()[1]):4.0%}", end="")
            elif line.startswith("error"):
                print(line.strip())
        proc.wait(); raw16.unlink(missing_ok=True); print()
        if proc.returncode:
            cache.unlink(missing_ok=True)
            raise RuntimeError(f"transcription failed ({proc.returncode})")
    data = json.loads(cache.read_text())["notes"]
    return sorted((Note(T0 + n["onset"], max(n["offset"] - n["onset"], 0.01), n["pitch"], 0.8)
                   for n in data if n["program"] in GUITAR_PROGRAMS and not n["is_drum"]),
                  key=lambda n: (n.time, n.midi))

compare = {}
for mode, instruments in MODES.items():
    print(f"{mode} ({SOURCE}, {T0:g}–{T1:g} s)")
    compare[mode] = transcribe_window(SOURCE, instruments)

In [ ]:
def describe(ns):
    onsets = np.array(sorted({round(n.time, 3) for n in ns}))
    gaps = np.diff(onsets) if len(onsets) > 2 else np.array([np.nan])
    return {"notes": len(ns), "onsets": len(onsets),
            "fast onsets (<0.2 s apart)": int((gaps < 0.2).sum()),
            "median gap (s)": round(float(np.median(gaps)), 3),
            "high notes (≥ E4)": sum(n.midi >= 64 for n in ns)}

compare_tabs = {mode: build_tab(ns, STEM, DRUMS, VOCALS) for mode, ns in compare.items()}
print(f"{'':<30}" + "".join(f"{m:>18}" for m in compare))
stats = {mode: describe(ns) for mode, ns in compare.items()}
for key in next(iter(stats.values())):
    print(f"{key:<30}" + "".join(f"{st[key]:>18}" for st in stats.values()))
def solo_list(tab):
    return ", ".join(f"{s['start']:.0f}–{s['end']:.0f}" for s in tab["solos"]) or "—"
print(f"{'solos found':<30}" + "".join(f"{solo_list(t):>18}" for t in compare_tabs.values()))

fig, axes = plt.subplots(2, len(compare), figsize=(16, 7), sharex=True, sharey=True, squeeze=False)
for col, (mode, ns) in enumerate(compare.items()):
    for row, (label, rows) in enumerate([("transcribed", as_rows(ns)),
                                         ("tab: auto part", as_rows(compare_tabs[mode]["parts"]["auto"]["notes"]))]):
        ax = axes[row][col]
        for t, d, m, v in rows:
            ax.add_patch(plt.Rectangle((t, m - 0.4), d, 0.8, lw=0, color=cmap(0.15 + 0.85 * min(v, 1))))
        for s in compare_tabs[mode]["solos"]:
            ax.axvspan(max(s["start"], T0), min(s["end"], T1), color="tab:red", alpha=0.07, lw=0)
        ax.set_title(f"{mode} — {label} ({len(rows)} notes)", loc="left", fontsize=10)
        ax.grid(axis="y", alpha=0.25)
lo = min((n.midi for ns in compare.values() for n in ns), default=40)
hi = max((n.midi for ns in compare.values() for n in ns), default=88)
axes[-1][0].set_xlim(T0, T1); axes[-1][0].set_ylim(lo - 1, hi + 1)
plt.tight_layout(); plt.show()

In [ ]:
y, rate = window_audio(SOURCE)
print(f"Source audio ({SOURCE})"); display(Audio(y, rate=rate))
for mode, ns in compare.items():
    play(f"{mode}: transcribed", render(ns))
    play(f"{mode}: tab (auto part)", render(compare_tabs[mode]["parts"]["auto"]["notes"]))